# FP32/FP64 FMM precision and backend comparison

One FP64 dense-direct plan is the accuracy baseline. The main order/depth sweep compares true FP32 and FP64 FMM execution on CPU-static; a representative case then compares every available FMM backend in both precisions. Exact CUDA direct is an optional runtime reference only.

In [ ]:
import gc, time
import cdfmm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SHAPE = (8, 8, 8)
REPEATS = 7
ORDERS = (2, 4, 6)
TREE_DEPTHS = (2, 3, 4)
REPRESENTATIVE_ORDER, REPRESENTATIVE_DEPTH = 4, 3
indices = np.indices(SHAPE, dtype=np.float64).reshape(3, -1).T
positions = (indices - (np.asarray(SHAPE) - 1) / 2) * 3.0e-8
particle = np.arange(len(positions), dtype=np.float64)
moments = 1.0e-21 * np.column_stack((0.7 + 0.2*np.sin(.17*particle), -0.3 + 0.2*np.cos(.11*particle), 0.4*np.sin(.07*particle + .3)))
identities = np.arange(len(positions), dtype=np.int32)
print(f'Particles: {len(positions):,}; CUDA: {cdfmm.cuda_available()}; oneMKL: {cdfmm.one_mkl_available()}')

In [ ]:
def construct(factory):
    start = time.perf_counter(); plan = factory()
    return plan, time.perf_counter() - start

def benchmark(evaluate):
    field = evaluate(); samples = []
    for _ in range(REPEATS):
        start = time.perf_counter(); field = evaluate(); samples.append(time.perf_counter() - start)
    return field, float(np.median(samples))

def errors(field, reference):
    difference = field.astype(np.float64) - reference
    absolute = np.linalg.norm(difference, axis=1); reference_norm = np.linalg.norm(reference, axis=1)
    meaningful = reference_norm > np.finfo(float).eps * reference_norm.max()
    return {'relative_l2': float(np.linalg.norm(difference)/np.linalg.norm(reference)), 'maximum_absolute': float(absolute.max()), 'maximum_relative': float(np.max(absolute[meaningful]/reference_norm[meaningful]))}

def options_for(order, depth, precision, backend, matrix_backend=None):
    options = cdfmm.UniformFmmOptions(); options.expansion_order = order; options.tree.max_level = depth
    options.precision = precision; options.backend = backend; options.fixed_target_source_indices = identities.tolist()
    if matrix_backend is not None: options.static_matrix_backend = matrix_backend
    return options

def measure_fmm(label, options):
    plan, construction_s = construct(lambda: cdfmm.UniformFmm(positions, positions, options))
    field, evaluation_s = benchmark(lambda: plan.evaluate(moments, target_source_indices=identities)['H'])
    assert np.isfinite(field).all(), f'{label} {options.precision} produced a non-finite field'
    host, device = dict(plan.static_plan_statistics), dict(plan.cuda_plan_statistics)
    row = {'label': label, 'precision': str(field.dtype), 'order': options.expansion_order, 'depth': options.tree.max_level, 'construction_s': construction_s, 'evaluation_s': evaluation_s, 'host_plan_mib': host['total_bytes']/2**20, 'persistent_device_mib': device['persistent_device_bytes']/2**20, 'evaluation_h2d_mib': device['evaluation_h2d_bytes']/2**20, 'evaluation_d2h_mib': device['evaluation_d2h_bytes']/2**20, **errors(field, H_dense)}
    return row, plan

## One FP64 dense-direct accuracy baseline

In [ ]:
dense_plan, dense_construction_s = construct(lambda: cdfmm.DenseDirectPlan(source_positions=positions, target_positions=positions, target_source_indices=identities.tolist(), static_precision='float64'))
H_dense, dense_evaluation_s = benchmark(lambda: dense_plan.evaluate(moments, backend=cdfmm.DenseDirectBackend.PORTABLE))
print(f'Dense construction {dense_construction_s:.3f} s; median evaluation {1e3*dense_evaluation_s:.3f} ms')

## Paired CPU-static order/depth sweep

Orders 2, 4, 6 and depths 2, 3, 4 are evaluated in both precisions. Host plan memory includes immutable operators, scratch, and mutable FMM state.

In [ ]:
sweep_rows = []
for depth in TREE_DEPTHS:
    for order in ORDERS:
        for precision in (cdfmm.StaticPrecision.FLOAT32, cdfmm.StaticPrecision.FLOAT64):
            options = options_for(order, depth, precision, cdfmm.ExecutionBackend.CPU_STATIC, cdfmm.StaticMatrixBackend.PORTABLE)
            row, plan = measure_fmm('CPU static portable', options); sweep_rows.append(row)
            del plan; gc.collect()
sweep = pd.DataFrame(sweep_rows)
sweep

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
for (depth, precision), selected in sweep.groupby(['depth', 'precision']):
    label = f'depth={depth}, {precision}'
    axes[0,0].semilogy(selected.order, selected.relative_l2, 'o-', label=label)
    axes[0,1].plot(selected.order, 1e3*selected.evaluation_s, 'o-', label=label)
    axes[1,0].plot(selected.order, selected.host_plan_mib, 'o-', label=label)
    axes[1,1].plot(selected.order, selected.construction_s, 'o-', label=label)
axes[0,0].set(title='Accuracy', ylabel='Relative L2 error'); axes[0,1].set(title='Runtime', ylabel='Median [ms]')
axes[1,0].set(title='Host plan memory', ylabel='MiB'); axes[1,1].set(title='Construction', ylabel='Seconds')
for axis in axes.flat: axis.set_xlabel('Order'); axis.set_xticks(ORDERS); axis.grid(alpha=.3); axis.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Representative backend comparison in both precisions

CPU-reference, portable CPU-static, oneMKL, CUDA-partial, and CUDA-full are included when available. Host memory, persistent device memory, and dynamic transfer bytes remain separate columns.

In [ ]:
backends = [('CPU reference', cdfmm.ExecutionBackend.CPU_REFERENCE, None), ('CPU static portable', cdfmm.ExecutionBackend.CPU_STATIC, cdfmm.StaticMatrixBackend.PORTABLE)]
if cdfmm.one_mkl_available(): backends.append(('CPU static oneMKL', cdfmm.ExecutionBackend.CPU_STATIC, cdfmm.StaticMatrixBackend.ONE_MKL))
if cdfmm.cuda_m2l_p2p_available(): backends.append(('CUDA partial', cdfmm.ExecutionBackend.CUDA_PARTIAL, None))
if cdfmm.cuda_full_available(): backends.append(('CUDA full', cdfmm.ExecutionBackend.CUDA_FULL, None))
backend_rows = []
for label, backend, matrix_backend in backends:
    for precision in (cdfmm.StaticPrecision.FLOAT32, cdfmm.StaticPrecision.FLOAT64):
        options = options_for(REPRESENTATIVE_ORDER, REPRESENTATIVE_DEPTH, precision, backend, matrix_backend)
        row, plan = measure_fmm(label, options); backend_rows.append(row)
        del plan; gc.collect()
backend_results = pd.DataFrame(backend_rows)
backend_results

## Optional exact CUDA direct runtime reference

In [ ]:
cuda_direct_row = None
if cdfmm.cuda_direct_available():
    cuda_direct, construction_s = construct(lambda: cdfmm.CudaDirectPlan(positions, positions, identities))
    field, evaluation_s = benchmark(lambda: cuda_direct.evaluate(moments)['H'])
    cuda_direct_row = {'construction_s': construction_s, 'evaluation_s': evaluation_s, **errors(field, H_dense)}
cuda_direct_row

## Interpretation

FP32 rows measure the change in the FMM itself: operators, multipoles, locals, scratch, near/far fields, transfers, and CUDA buffers are single precision. Positions and analytical construction remain double precision; completed operators are quantised once without a persistent FP64 copy. The dense plan is only the accuracy baseline.